# Yuki VTuber - Servidor de Voz (GPT-SoVITS)

Execute as celulas em ordem. Na celula 4 faca upload do audio de referencia.

In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Nenhuma')

In [ ]:
import os
if not os.path.exists('/content/GPT-SoVITS'):
    !git clone https://github.com/RVC-Boss/GPT-SoVITS /content/GPT-SoVITS
os.chdir('/content/GPT-SoVITS')
!pip install -q -r requirements.txt
!pip install -q fastapi uvicorn
print('Instalacao concluida!')

In [ ]:
import os
pd = '/content/GPT-SoVITS/GPT_SoVITS/pretrained_models'
os.makedirs(pd+'/gsv-v2final-pretrained', exist_ok=True)
models = {
    'gsv-v2final-pretrained/s2G2333k.pth': 'https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/gsv-v2final-pretrained/s2G2333k.pth',
    'gsv-v2final-pretrained/s1bert25hz-5kh-longer-epoch=12-step=369668.ckpt': 'https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/gsv-v2final-pretrained/s1bert25hz-5kh-longer-epoch%3D12-step%3D369668.ckpt',
}
for p, url in models.items():
    fp = f'{pd}/{p}'
    if not os.path.exists(fp):
        print(f'Baixando {p}...')
        os.system(f'wget -q -O "{fp}" "{url}"')
    else:
        print(f'OK {p}')
for m in ['chinese-roberta-wwm-ext-large', 'chinese-hubert-base']:
    mp = f'{pd}/{m}'
    if not os.path.exists(mp):
        print(f'Clonando {m}...')
        os.system(f'git clone --depth 1 https://huggingface.co/hfl/{m} {mp}')
    else:
        print(f'OK {m}')
print('Modelos prontos!')

In [ ]:
from google.colab import files
import os
os.makedirs('/content/ref_audio', exist_ok=True)
print('Faca upload do arquivo .wav de referencia da Yuki')
print('Local: C:\\GPT-SoVITS\\...\\output\\slicer_opt\\')
print('Arquivo: audio [vocals].mp3_0000000000_0000183040.wav')
uploaded = files.upload()
for fn, data in uploaded.items():
    dest = f'/content/ref_audio/{fn}'
    open(dest, 'wb').write(data)
    open('/content/ref_audio_path.txt', 'w').write(dest)
    print(f'Salvo: {dest}')

In [ ]:
import os, subprocess, time
os.chdir('/content/GPT-SoVITS')
ref = open('/content/ref_audio_path.txt').read().strip()
os.makedirs('GPT_SoVITS/configs', exist_ok=True)
config = f"""default:
  device: cuda
  is_half: true
  version: v2
  t2s_weights_path: GPT_SoVITS/pretrained_models/gsv-v2final-pretrained/s1bert25hz-5kh-longer-epoch=12-step=369668.ckpt
  vits_weights_path: GPT_SoVITS/pretrained_models/gsv-v2final-pretrained/s2G2333k.pth
  bert_base_path: GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large
  cnhuhbert_base_path: GPT_SoVITS/pretrained_models/chinese-hubert-base
  ref_audio_path: {ref}
  prompt_text: Vamos la menino segura essa espada
  prompt_lang: en
  text_lang: en
  text_split_method: cut5
  batch_size: 1
  media_type: wav
  streaming_mode: false
"""
open('GPT_SoVITS/configs/tts_infer.yaml', 'w').write(config)
proc = subprocess.Popen(
    ['python', 'api_v2.py', '-a', '0.0.0.0', '-p', '9880', '-c', 'GPT_SoVITS/configs/tts_infer.yaml'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
print('Iniciando servidor...')
for i in range(90):
    l = proc.stdout.readline()
    if l.strip(): print(l.strip())
    if 'startup complete' in l or 'Uvicorn running' in l:
        print('Servidor ativo na porta 9880!')
        break
    time.sleep(1)

In [ ]:
import subprocess, time, re, os
os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared')
tp = subprocess.Popen(['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:9880'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print('Aguardando URL do tunel...')
for i in range(40):
    l = tp.stdout.readline()
    if 'trycloudflare.com' in l:
        m = re.search(r'https://[\w-]+\.trycloudflare\.com', l)
        if m:
            url = m.group(0)
            print(f'SERVIDOR ATIVO!')
            print(f'URL: {url}')
            print(f'Cole no conf.yaml local:')
            print(f'  api_url: "{url}/tts"')
            break
    time.sleep(1)

In [ ]:
import time
print('Sessao ativa - nao feche esta celula')
while True:
    time.sleep(60)
    print(f'Online {time.strftime("%H:%M:%S")}')